# 過擬合與泛化

對應 DLI：02、03。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/06_overfitting_generalization.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/06-overfitting-generalization.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

## 學習目標與先備知識

先理解訓練／驗證切分與 MSE。學完能從訓練及驗證結果提出診斷假設，且不把「模型較大」直接等同「模型較好」。



## 1. 背答案與學規律有什麼不同？

**來源摘要 [S4](https://scikit-learn.org/stable/modules/learning_curve.html)：**偏差描述估計方法的系統性誤差，變異描述對訓練資料改變的敏感程度。訓練與驗證曲線可協助觀察模型複雜度是否合適，但不能取代獨立測試。

**本教材自編比喻：**學生把題目及答案一字不漏背下來，原題能全對；只要換題就失敗。這類比過擬合，但不能用這個比喻取代實際資料評估。



## 2. 資料可以想成規律加雜訊

$$y=f(x)+\epsilon$$

f(x) 是希望學到的規律，$\epsilon$ 表示未由輸入完整說明的變動。本課自編資料使用 sin(3x) 加上常態雜訊；這是已知生成機制，真實專案通常不知道 f。

| 狀態 | 本課曲線的可能表現 | 該查什麼 |
|---|---|---|
| 欠擬合 | 訓練與驗證誤差都偏高 | 特徵是否足夠、模型是否過於受限 |
| 過擬合 | 訓練誤差低、驗證誤差高 | 資料量、雜訊、複雜度與正則化 |
| 較合理的擬合 | 訓練與驗證結果符合需求 | 仍需測試資料與部署檢查 |

這是診斷線索，不是只看一張圖就能定論。訓練不足、資料前處理錯誤及分布差異，也會影響曲線。



## 3. 手算理解「差距」

本教材自編三個模型：

| 模型 | 訓練 MSE | 驗證 MSE |
|---|---:|---:|
| A | 12 | 13 |
| B | 3 | 4 |
| C | 0.2 | 9 |

若任務與切分相同，B 在這次驗證比較較好。C 的訓練／驗證差距為 8.8，大於 B 的 1；但只報差距也不夠，A 的差距同樣只有 1，絕對誤差仍較高。



## 4. 正則化是在做什麼？

可把學習目標想成：

$$\text{訓練資料損失}+\lambda\times\text{模型複雜度懲罰}$$

本課 Ridge 的懲罰與係數平方和有關。較大的懲罰會限制模型自由度，但太大也可能欠擬合。這不是「把分數作弊扣掉」，而是學習目標本身改變。

自編練習依序嘗試多項式次數 1、3、15；固定相同資料後再改懲罰，才能看出各因素的作用。此一維曲線不能直接證明 CNN 的所有行為。



## 5. 不同橫軸代表不同問題

- 模型複雜度：改深度或多項式次數，問容量是否合適。
- 訓練資料量：逐步加資料，問更多資料是否有幫助。
- Epoch：看訓練過程，問是否訓練不足或開始過擬合。

DLI 常畫的是 Epoch 曲線；不要把它與「訓練樣本數」曲線當成同一種實驗。Early stopping 可依驗證表現選停止點，之後仍用獨立測試評估。

## 自我檢核

1. 訓練 Accuracy 100%，驗證 60%，應先宣布成功嗎？
2. 訓練及驗證都差時，增加資料一定有用嗎？
3. 用驗證集挑出最佳模型後，還需要測試集嗎？

### 參考解答

1. 不應。先查洩漏、資料差異、過擬合及流程問題。
2. 不一定；特徵不足、模型受限或實作錯誤也可能是原因。
3. 需要；驗證集已參與選擇，不能當作完全未受選擇影響的最後評估。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
rng = np.random.default_rng(42)
x_train = np.linspace(-1, 1, 20).reshape(-1, 1)
y_train = np.sin(3*x_train[:, 0]) + rng.normal(0, .25, 20)
x_val = np.linspace(-1, 1, 100).reshape(-1, 1)
y_val = np.sin(3*x_val[:, 0]) + rng.normal(0, .25, 100)
for degree in [1, 3, 15]:
    model = make_pipeline(PolynomialFeatures(degree), Ridge(alpha=1e-8))
    model.fit(x_train, y_train)
    print(degree, 'train MSE=', mean_squared_error(y_train, model.predict(x_train)),
          'validation MSE=', mean_squared_error(y_val, model.predict(x_val)))
    plt.plot(x_val, model.predict(x_val), label=f'degree {degree}')
plt.scatter(x_train, y_train); plt.ylim(-3, 3); plt.legend(); plt.show()

## 如何解讀

比較曲線與兩組誤差，避免只挑訓練誤差最低者。具體排序受樣本與雜訊影響，應多次重複；此驗證集可選模型，正式測試集仍需另外保留。

## 練習

改變樣本數及 Ridge alpha，記錄對訓練與驗證誤差的影響，每次只改一個因素。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S4**：[scikit-learn，Validation curves](https://scikit-learn.org/stable/modules/learning_curve.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
